# Generate `threshold.json` for META-CXR

Notebook này tạo `threshold.json` mới từ **validation set** của dataset đang dùng. Không tune trên test set.

Kaggle datasets cần attach:
- `mimic-cxr-checkpoint`: chứa checkpoint model đã train.
- `mimic-cxr-jpg-lite`: chứa ảnh + CheXpert CSV.
- `mimic-cxr-p10-processed`: chứa `train.csv`, `val.csv`, `test.csv`.
- Source/dataset chứa code `META-CXR` nếu notebook không nằm sẵn trong repo.

Output:
- `/kaggle/working/threshold.json`
- `/kaggle/working/threshold_search_details.csv`

In [ ]:
# Kaggle: bật Internet nếu môi trường chưa có đủ dependency/cache model.
# Không pin transformers==4.30.2 ở Kaggle mới vì nó kéo tokenizers<0.14 và có thể phải build từ source.
# Code Qformer hiện đã fallback sang transformers.pytorch_utils nên dùng transformers sẵn có của Kaggle là ổn.
!pip install -q \
  "numpy<2" \
  "opencv-python<4.10" \
  "omegaconf==2.3.0" \
  iopath timm pandas scikit-image accelerate sentencepiece protobuf \
  iterative-stratification einops fairscale pycocoevalcap webdataset decord \
  ftfy regex hi-ml-multimodal torchinfo

In [ ]:
import os
import sys
import shutil
from pathlib import Path

WORK_DIR = Path('/kaggle/working')
INPUT_DIR = Path('/kaggle/input')

def is_meta_cxr_project(path: Path) -> bool:
    return (path / 'model' / 'lavis').exists() and (path / 'pretraining').exists()

project_candidates = [Path.cwd(), WORK_DIR / 'META-CXR']
if INPUT_DIR.exists():
    for root in INPUT_DIR.glob('*'):
        project_candidates.extend([root, root / 'META-CXR'])

source_project = next((p for p in project_candidates if is_meta_cxr_project(p)), None)
if source_project is None:
    raise FileNotFoundError('Không tìm thấy code META-CXR. Hãy attach dataset/source chứa thư mục META-CXR.')

PROJECT_DIR = WORK_DIR / 'META-CXR'
if source_project.resolve() != PROJECT_DIR.resolve():
    if PROJECT_DIR.exists() and is_meta_cxr_project(PROJECT_DIR):
        pass
    else:
        ignore = shutil.ignore_patterns('.git', 'wandb', '__pycache__', '*.pyc', 'output', 'outputs', 'checkpoints')
        shutil.copytree(source_project, PROJECT_DIR, dirs_exist_ok=True, ignore=ignore)

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'model'))

def first_existing(paths):
    for item in paths:
        path = Path(item)
        if path.exists():
            return path
    raise FileNotFoundError('Không tìm thấy path nào trong: ' + ', '.join(map(str, paths)))

IMAGE_ROOT = first_existing([
    '/kaggle/input/mimic-cxr-jpg-lite',
    '/kaggle/input/datasets/mimic-cxr-jpg-lite',
])
PROCESSED_ROOT = first_existing([
    '/kaggle/input/mimic-cxr-p10-processed',
    '/kaggle/input/datasets/mimic-cxr-p10-processed',
])
CHECKPOINT_ROOT = first_existing([
    '/kaggle/input/mimic-cxr-checkpoint',
    '/kaggle/input/meta-cxr-checkpoint',
    '/kaggle/input/meta-cxr-checkpoints',
])

(PROJECT_DIR / 'configs').mkdir(exist_ok=True)
(PROJECT_DIR / 'configs' / 'env_config.yaml').write_text(f'''paths:
  data_root: "{IMAGE_ROOT}"
  mimic_cxr_jpg_root: "{IMAGE_ROOT}"
  split_csv: "{IMAGE_ROOT}/mimic-cxr-2.0.0-split.csv"
  reports_csv: "/kaggle/working/mimic_cxr_cleaned.csv"
  chexpert_csv: "{IMAGE_ROOT}/mimic-cxr-2.0.0-chexpert.csv"
  metadata_csv: "{IMAGE_ROOT}/mimic-cxr-2.0.0-metadata.csv"
  processed_dir: "{PROCESSED_ROOT}"
  processed_train_csv: "{PROCESSED_ROOT}/train.csv"
  processed_val_csv: "{PROCESSED_ROOT}/val.csv"
  processed_test_csv: "{PROCESSED_ROOT}/test.csv"
  output_dir: "/kaggle/working/output"
  checkpoint_dir: "{CHECKPOINT_ROOT}"
wandb:
  entity: ""
  project: "meta-cxr-encoder-comparison"
java:
  home: "/usr/lib/jvm/java-11-openjdk-amd64"
  path: "/usr/lib/jvm/java-11-openjdk-amd64/bin:"
''')

print('PROJECT_DIR    =', PROJECT_DIR)
print('IMAGE_ROOT     =', IMAGE_ROOT)
print('PROCESSED_ROOT =', PROCESSED_ROOT)
print('CHECKPOINT_ROOT=', CHECKPOINT_ROOT)

## Chọn checkpoint dùng để tune threshold

Thông thường nên dùng checkpoint của model cuối cùng bạn dùng để inference/report generation. Nếu bạn muốn tune threshold cho từng encoder riêng, chạy notebook này nhiều lần và đổi `RUN_NAME`.

In [ ]:
# Đổi RUN_NAME nếu muốn tạo threshold cho encoder khác.
RUN_NAME = '07_all_three'

# Dùng validation set để tune threshold. Không đổi sang test nếu dùng cho báo cáo kết quả.
SPLIT = 'val'

# Lưới threshold. 0.01 đủ mịn cho validation nhỏ; có thể giảm step nếu cần.
THRESHOLD_GRID = [round(x / 100, 2) for x in range(1, 100)]

EVAL_BATCH_SIZE = 4
NUM_WORKERS = 2

In [ ]:
import gc
import json
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

import model.lavis.tasks as tasks
from model.lavis.common.config import Config
from model.lavis.common.registry import registry

# Registration imports.
from model.lavis.common.optims import LinearWarmupCosineLRScheduler, LinearWarmupStepLRScheduler
from model.lavis.datasets.builders import *
from model.lavis.models import *
from model.lavis.processors import *
from model.lavis.tasks import *
from model.lavis.data.ReportDataset import MIMIC_CXR_Dataset
from local_config import VIS_ROOT

registry.mapping['paths']['cache_root'] = '.'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

CHEXPERT_COLS = [
    'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
    'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis',
    'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices'
]

# Index class trong code hiện tại: negative=0, positive=1, uncertain=2.
# No Finding và Support Devices có thể chỉ có 2 class trong threshold cũ, nhưng model vẫn xuất 3 class.
CLASS_MAP = {'negative': 0, 'positive': 1, 'uncertain': 2}

print('DEVICE =', DEVICE)

In [ ]:
def find_checkpoint(run_name: str) -> Path:
    best = [p for p in CHECKPOINT_ROOT.rglob('checkpoint_best.pth') if run_name in str(p)]
    if best:
        return sorted(best, key=lambda p: len(str(p)))[0]
    last = [p for p in CHECKPOINT_ROOT.rglob('checkpoint_last.pth') if run_name in str(p)]
    if last:
        print(f'WARNING: {run_name}: checkpoint_best.pth not found, using checkpoint_last.pth')
        return sorted(last, key=lambda p: len(str(p)))[0]
    raise FileNotFoundError(f'Không tìm thấy checkpoint_best/last cho run {run_name} trong {CHECKPOINT_ROOT}')

def build_cfg(run_name: str):
    cfg_path = PROJECT_DIR / 'pretraining' / 'configs' / 'encoder_comparison' / f'{run_name}.yaml'
    args = SimpleNamespace(cfg_path=str(cfg_path), options=None)
    return Config(args)

def build_model_for_run(run_name: str, checkpoint_path: Path):
    cfg = build_cfg(run_name)
    task = tasks.setup_task(cfg)
    model = task.build_model(cfg)
    ckpt = torch.load(checkpoint_path, map_location='cpu')
    state_dict = ckpt['model'] if isinstance(ckpt, dict) and 'model' in ckpt else ckpt
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print(f'{run_name}: loaded {checkpoint_path.name}; missing={len(missing)}, unexpected={len(unexpected)}')
    model.to(DEVICE)
    model.eval()
    return cfg, model

def make_loader(cfg, split: str):
    dataset = MIMIC_CXR_Dataset(
        vis_processor=None,
        text_processor=None,
        vis_root=VIS_ROOT,
        split=split,
        cfg=cfg,
        truncate=None,
    )
    return DataLoader(
        dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == 'cuda'),
    )

@torch.no_grad()
def predict_logits_with_text(model, batch):
    image = batch['image'].to(DEVICE, non_blocking=True)
    text = batch['text_output']

    cnn_patches, vit_patches, swin_patches, _ = model._encode_image_streams(image, apply_aug=False)
    text_tokens = model.tokenizer(
        text,
        padding='max_length',
        truncation=True,
        max_length=model.max_txt_len,
        return_tensors='pt',
    ).to(DEVICE)
    text_output = model.Qformer.bert(
        text_tokens.input_ids,
        attention_mask=text_tokens.attention_mask,
        return_dict=True,
    )
    logits, _, _, _, _ = model.mhcac(
        cnn_patches=cnn_patches,
        vit_patches=vit_patches,
        swin_patches=swin_patches,
        text_embeddings=text_output.last_hidden_state,
        labels=None,
    )
    return logits

In [ ]:
checkpoint_path = find_checkpoint(RUN_NAME)
cfg, model = build_model_for_run(RUN_NAME, checkpoint_path)
loader = make_loader(cfg, SPLIT)

all_probs = []
all_labels = []

for batch in tqdm(loader, desc=f'{RUN_NAME}:{SPLIT}'):
    logits = predict_logits_with_text(model, batch)
    probs = torch.softmax(logits, dim=-1)
    all_probs.append(probs.cpu().numpy())
    all_labels.append(batch['classification_labels'].cpu().numpy())

probs = np.concatenate(all_probs, axis=0)      # [N, 14, 3]
labels = np.concatenate(all_labels, axis=0)    # [N, 14]

print('probs shape =', probs.shape)
print('labels shape =', labels.shape)

del model, loader
gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

In [ ]:
def best_threshold_for_binary(y_true_binary, scores, grid):
    best_t = 0.5
    best_f1 = -1.0
    for t in grid:
        y_pred_binary = (scores >= t).astype(int)
        score = f1_score(y_true_binary, y_pred_binary, zero_division=0)
        if score > best_f1:
            best_f1 = float(score)
            best_t = float(t)
    return best_t, best_f1

thresholds = {}
detail_rows = []

for task_idx, abnormality in enumerate(CHEXPERT_COLS):
    thresholds[abnormality] = {}
    y_task = labels[:, task_idx]
    for class_name, class_idx in CLASS_MAP.items():
        # Nếu validation set không có class này cho abnormality đó thì bỏ qua.
        y_true_binary = (y_task == class_idx).astype(int)
        positives = int(y_true_binary.sum())
        if positives == 0:
            continue
        scores = probs[:, task_idx, class_idx]
        best_t, best_f1 = best_threshold_for_binary(y_true_binary, scores, THRESHOLD_GRID)
        thresholds[abnormality][class_name] = best_t
        detail_rows.append({
            'abnormality': abnormality,
            'class': class_name,
            'threshold': best_t,
            'val_f1': best_f1,
            'positive_count': positives,
            'n': len(y_true_binary),
        })

threshold_path = Path('/kaggle/working/threshold.json')
details_path = Path('/kaggle/working/threshold_search_details.csv')

threshold_path.write_text(json.dumps(thresholds, indent=4), encoding='utf-8')
details = pd.DataFrame(detail_rows)
details.to_csv(details_path, index=False)

print('Saved', threshold_path)
print('Saved', details_path)
display(details.sort_values(['abnormality', 'class']).reset_index(drop=True))
print(json.dumps(thresholds, indent=4)[:2000])

## Ghi chú sử dụng

- File `/kaggle/working/threshold.json` sinh từ validation set, phù hợp hơn với subset/dataset hiện tại so với threshold cũ từ dataset lớn hơn.
- Không dùng file này để báo cáo test F1 nếu bạn đã tune trực tiếp trên test.
- Nếu inference dùng `07_all_three`, nên tạo threshold với `RUN_NAME = '07_all_three'`.
- Nếu muốn mỗi encoder có threshold riêng, chạy lại notebook cho từng `RUN_NAME` và lưu tên file riêng, ví dụ `threshold_01_biovil_only.json`.